In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pyspark.sql.functions as F
from pyspark.sql import *
catalog_name = 'ecommerce'

In [0]:
df = spark.read.option("header", "True").option("inferSchema", "True").csv("/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv")
df.schema

In [0]:
order_items_schema = StructType([
  StructField('dt', StringType(), True), 
  StructField('order_ts', StringType(), True), 
  StructField('customer_id', StringType(), True), 
  StructField('order_id', StringType(), True), 
  StructField('item_seq', StringType(), True), 
  StructField('product_id', StringType(), True), 
  StructField('quantity', StringType(), True), 
  StructField('unit_price_currency', StringType(), True), 
  StructField('unit_price', StringType(), True), 
  StructField('discount_pct', StringType(), True), 
  StructField('tax_amount', StringType(), True), 
  StructField('channel', StringType(), True),
  StructField('coupon_code', StringType(), True)])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv"

df = spark.read.option('header', "true").option("demlimiter", ",").schema(order_items_schema).csv(raw_data_path)

# Add metadata columns
df = df.withColumn("file_name", F.col("_metadata.file_path")) \
       .withColumn("ingested_timestamp", F.current_timestamp())

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_order_items")

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_order_items")

In [0]:
df_silver = df_bronze.dropDuplicates(["order_id", "item_seq"])

df_silver = df_silver.withColumn("quantity", F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int"))
df_silver = df_silver.withColumn("unit_price", F.regexp_replace("unit_price", "[$]", "").cast("double"))
df_silver = df_silver.withColumn("discount_pct", F.regexp_replace("discount_pct", "%", "").cast("double"))
df_silver = df_silver.withColumn("coupon_code", F.lower(F.trim(F.col("coupon_code"))))
df_silver = df_silver.withColumn("channel", F.when(F.col("channel") == "web", "Website").when(F.col("channel") == "app", "Mobile").otherwise(F.col("channel")))

In [0]:
# Transformation: datatype conversions
# 1) Convert dt (string → date)
df_silver = df_silver.withColumn(
    "dt",
    F.to_date("dt", "yyyy-MM-dd")     
)

# 2) Convert order_ts (string → timestamp)
df_silver = df_silver.withColumn(
    "order_ts",
    F.coalesce(
        F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"),  # matches 2025-08-01 22:53:52
        F.to_timestamp("order_ts", "dd-MM-yyyy HH:mm")      # fallback for 01-08-2025 22:53
    )
)


# 3) Convert item_seq (string → integer)
df_silver = df_silver.withColumn(
    "item_seq",
    F.col("item_seq").cast("int")
)

# 4) Convert tax_amount (string → double, strip non-numeric characters)
df_silver = df_silver.withColumn(
    "tax_amount",
    F.regexp_replace("tax_amount", r"[^0-9.\-]", "").cast("double")
)


#Transformation : Add processed time 
df_silver = df_silver.withColumn(
    "processed_time", F.current_timestamp()
)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_order_items")

In [0]:
df = spark.table(f"{catalog_name}.silver.slv_order_items")

In [0]:
# 1) Add gross amount
df = df.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
    )

# 2) Add discount_amount (discount_pct is already numeric, e.g., 21 -> 21%)
df = df.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

# 3) Add sale_amount = gross - discount
df = df.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

# add date id
df = df.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key

# Coupon flag
#  coupon flag = 1 if coupon_code is not null else 0
df = df.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)

df.limit(5).display()    

In [0]:
# --- 1) Define your fixed FX rates (as of 2025-10-15, like your PBI note) ---
## Include currency API here
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29,
}

rates = [(k, float(v)) for k, v in fx_rates.items()]
rates_df = spark.createDataFrame(rates, ["currency", "inr_rate"])
rates_df.show()

In [0]:
df = (
    df
    .join(
        rates_df,
        rates_df.currency == F.upper(F.trim(F.col("unit_price_currency"))),
        "left"
    )
    .withColumn("sale_amount_inr", F.col("sale_amount") * F.col("inr_rate"))
    .withColumn("sale_amount_inr", F.ceil(F.col("sale_amount_inr")))
)

In [0]:
orders_gold_df = df.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")
)

In [0]:
# Write raw data to the gold layer (catalog: ecommerce, schema: gold, table: gld_fact_order_items)
orders_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")